# XD2-Net — 01: Data Preparation

Loads the raw event log, performs train/test split, role discovery, feature encoding,
vectorization, and saves per-bucket artifacts for use in the training notebook.

Run this once before training.

In [ ]:
import os, sys

# ── Colab / local detection ──────────────────────────────────────────────────
try:
    import google.colab
    IN_COLAB = True
except ImportError:
    IN_COLAB = False

if IN_COLAB:
    from google.colab import drive
    drive.mount('/content/drive')
    REPO_ROOT = '/content/drive/MyDrive/xd2-net'
    %pip install -e {REPO_ROOT} --quiet
else:
    REPO_ROOT = os.path.abspath('..')
    sys.path.insert(0, REPO_ROOT)

os.chdir(REPO_ROOT)
print('REPO_ROOT:', REPO_ROOT)

In [ ]:
import pickle
import logging
import numpy as np
import pandas as pd

from xd2net.config import load_config
from xd2net.data.preprocessing import (
    RoleDiscovery, balance_dataset, create_index,
    extract_prefixes, normalize_events, reformat_events,
    split_train_test, trace_stats, vectorize,
)
from xd2net.data.feature_engineering import generate_inputs, align_exp_dims

logging.basicConfig(level=logging.INFO, format='%(levelname)s: %(message)s')

# Load config — override csv_path for Colab if needed
cfg = load_config('configs/bpic2012.yaml')
if IN_COLAB:
    cfg.data.csv_path = f'{REPO_ROOT}/data/BPIC_2012_OUTCOME.csv'

print('Config loaded:', cfg.data.csv_path)

In [ ]:
# ── Load raw CSV ─────────────────────────────────────────────────────────────
df = pd.read_csv(cfg.data.csv_path)
print(df.shape)
df.head()

In [ ]:
# ── Rename & index columns ────────────────────────────────────────────────────
df = df.rename(columns={
    cfg.data.caseid_col: 'CASEID',
    cfg.data.activity_col: 'MILESTONE',
})
df['ACTIVITY'] = df['MILESTONE']
df['oc_index'] = (df[cfg.data.outcome_col] == cfg.data.approved_label).astype(int)

# Sort by time and create activity index within each case
time_col = 'Complete Timestamp' if 'Complete Timestamp' in df.columns else df.columns[2]
df = df.sort_values(['CASEID', time_col])
df['ACTIVITY_INDEX'] = df.groupby('CASEID').cumcount() + 1

print('Outcome distribution:')
print(df.groupby('CASEID')[cfg.data.outcome_col].first().value_counts())

In [ ]:
# ── Train / test split ────────────────────────────────────────────────────────
df_train_full, df_test_full = split_train_test(df, cfg.data.test_split, seed=42)
print(f'Train cases: {df_train_full["CASEID"].nunique()}  Test cases: {df_test_full["CASEID"].nunique()}')

In [ ]:
# ── Role discovery (fit on train, apply to test) ──────────────────────────────
role_disc = RoleDiscovery(n_clusters=cfg.data.role_discovery_clusters)
df_train_full = role_disc.fit(df_train_full)
df_test_full  = role_disc.transform(df_test_full)
print('Roles:', df_train_full['ROLE'].value_counts().to_dict())

In [ ]:
# ── Numerical normalization (fit on train only) ───────────────────────────────
all_num = cfg.data.num_dynamic + cfg.data.num_static
df_train_full, scalers = normalize_events(df_train_full, all_num, cfg.data.norm_method)
df_test_full,  _       = normalize_events(df_test_full,  all_num, cfg.data.norm_method, scalers=scalers)
print('Scalers fitted for:', list(scalers.keys()))

In [ ]:
# ── Categorical index encoding ────────────────────────────────────────────────
indexes = {'dynamic_indexes': {}, 'static_indexes': {}}

for feat in cfg.data.cat_dynamic:
    idx = create_index(df_train_full, feat, reserve_zero=True)
    indexes['dynamic_indexes'][f'{feat}_index'] = idx
    df_train_full[f'{feat}_index'] = df_train_full[feat].map(idx).fillna(0).astype(int)
    df_test_full[f'{feat}_index']  = df_test_full[feat].map(idx).fillna(0).astype(int)

for feat in cfg.data.cat_static:
    idx = create_index(df_train_full, feat, reserve_zero=True)
    indexes['static_indexes'][f'{feat}_index'] = idx
    df_train_full[f'{feat}_index'] = df_train_full[feat].map(idx).fillna(0).astype(int)
    df_test_full[f'{feat}_index']  = df_test_full[feat].map(idx).fillna(0).astype(int)

indexes['oc_index'] = {0: 'A_DECLINED', 1: 'A_APPROVED'}
print('Activities:', len(indexes['dynamic_indexes']['ACTIVITY_index']))
print('Roles:', len(indexes['dynamic_indexes']['ROLE_index']))

In [ ]:
# ── Per-bucket vectorization ──────────────────────────────────────────────────
artifacts = {}  # keyed by bucket_key, e.g. '1_15'

for n_size in cfg.data.prefix_lengths:
    bucket_key = f'1_{n_size}'

    df_b_train = extract_prefixes(df_train_full, n_size)
    df_b_test  = extract_prefixes(df_test_full,  n_size)

    if df_b_train['CASEID'].nunique() < 10:
        print(f'{bucket_key}: too few cases, skipping')
        continue

    if cfg.data.balance_train:
        df_b_train = balance_dataset(df_b_train, cfg.data.outcome_col,
                                     cfg.data.approved_label, 'CASEID')

    log_train = reformat_events(df_b_train, indexes, cfg.data.cat_dynamic,
                                cfg.data.num_dynamic, cfg.data.cat_static, cfg.data.num_static)
    log_test  = reformat_events(df_b_test,  indexes, cfg.data.cat_dynamic,
                                cfg.data.num_dynamic, cfg.data.cat_static, cfg.data.num_static)

    max_len_tr, _ = trace_stats(log_train)
    max_len_te, _ = trace_stats(log_test)
    max_len = max(max_len_tr, max_len_te)

    vec_train = vectorize(log_train, max_len, cfg.data.cat_dynamic, cfg.data.num_dynamic,
                          cfg.data.cat_static, cfg.data.num_static)
    vec_test  = vectorize(log_test,  max_len, cfg.data.cat_dynamic, cfg.data.num_dynamic,
                          cfg.data.cat_static, cfg.data.num_static)

    inp_tr, out_tr, exp_dim, feat_names = generate_inputs(
        vec_train, indexes, max_len, cfg.data.cat_dynamic, cfg.data.num_dynamic,
        cfg.data.cat_static, cfg.data.num_static)
    inp_te, out_te, _, feat_names_te = generate_inputs(
        vec_test,  indexes, max_len, cfg.data.cat_dynamic, cfg.data.num_dynamic,
        cfg.data.cat_static, cfg.data.num_static)

    inp_tr, inp_te, feat_names = align_exp_dims(inp_tr, inp_te, feat_names, feat_names_te)

    artifacts[bucket_key] = {
        'inputs_train': inp_tr,  'outputs_train': out_tr,
        'lengths_train': vec_train['lengths'], 'caseids_train': vec_train['caseid'],
        'inputs_test':  inp_te,  'outputs_test':  out_te,
        'lengths_test':  vec_test['lengths'],  'caseids_test':  vec_test['caseid'],
        'exp_dim': inp_tr['input_exp'].shape[1],
        'max_len': max_len,
        'feature_names': feat_names,
        'dim_dynamic': inp_tr['input_dynamic'].shape[2],
        'dim_static':  inp_tr['input_static'].shape[1],
    }
    print(f'{bucket_key}: train={vec_train["outcome"].shape[0]}  test={vec_test["outcome"].shape[0]}  exp_dim={inp_tr["input_exp"].shape[1]}')

print('\nDone. Buckets prepared:', list(artifacts.keys()))

In [ ]:
# ── Save artifacts ────────────────────────────────────────────────────────────
artifacts_dir = os.path.join(cfg.output.workspace_dir, 'artifacts')
os.makedirs(artifacts_dir, exist_ok=True)

for bucket_key, art in artifacts.items():
    path = os.path.join(artifacts_dir, f'bucket_{bucket_key}.pkl')
    with open(path, 'wb') as f:
        pickle.dump(art, f)

# Also save indexes and scalers
with open(os.path.join(artifacts_dir, 'indexes.pkl'), 'wb') as f:
    pickle.dump(indexes, f)
with open(os.path.join(artifacts_dir, 'scalers.pkl'), 'wb') as f:
    pickle.dump(scalers, f)

print('Artifacts saved to', artifacts_dir)

In [ ]:
# ── Verification: reload and check shapes ────────────────────────────────────
for bucket_key in list(artifacts.keys())[:3]:
    path = os.path.join(artifacts_dir, f'bucket_{bucket_key}.pkl')
    with open(path, 'rb') as f:
        a = pickle.load(f)
    print(f"{bucket_key}: dynamic={a['inputs_train']['input_dynamic'].shape}  "
          f"static={a['inputs_train']['input_static'].shape}  "
          f"exp={a['inputs_train']['input_exp'].shape}")